# 使用 Cognee 构建具有持久记忆的 AI 智能体

本笔记本演示如何使用 [**cognee**](https://www.cognee.ai/) 构建具有复杂记忆能力的智能 AI 智能体 - 一个开源 AI 记忆系统，结合了知识图谱、语义搜索和会话管理，创建具有上下文感知能力的 AI 系统。

## 🎯 学习目标

通过本教程结束时，您将了解如何：
- **构建由嵌入支持的知识图谱**：将非结构化文本转换为结构化、可查询的知识
- **实现会话记忆**：创建具有自动上下文保留的多轮对话
- **持久化对话**：可选地将重要交互存储在长期记忆中以供将来参考
- **使用自然语言查询**：在新对话中访问和利用历史上下文
- **可视化记忆**：探索智能体知识图谱中的关系

## 🏗️ 您将构建什么

在本教程中，我们将创建一个具有持久记忆的 **编码助手**，它：

### 1. **知识库构建**
   - 摄取开发者资料和专业知识信息
   - 处理 Python 编程原则和最佳实践
   - 存储开发者和 AI 助手之间的历史对话

### 2. **会话感知对话**
   - 在同一会话中跨多个问题维护上下文
   - 自动缓存每个问题/答案对以实现高效检索
   - 基于对话历史提供连贯、上下文相关的响应

### 3. **长期记忆**
   - 将重要对话持久化到长期记忆中
   - 从知识库和过去的会话中检索相关记忆以指导新的交互
   - 构建随时间改进的不断增长的知识库

### 4. **智能记忆检索**
   - 使用图谱感知语义搜索在所有存储的知识中找到相关信息
   - 按数据子组过滤搜索（开发者信息 vs. 原则）
   - 结合多个数据源以提供全面的答案

## 📋 前提条件和设置

### 系统要求

开始之前，请确保您有：

1. **Python 环境**
   - Python 3.9 或更高版本
   - 虚拟环境（推荐）
   
2. **Redis 缓存**（会话管理所需）
   - 本地 Redis：`docker run -d -p 6379:6379 redis`
   - 或使用托管 Redis 服务
   
3. **LLM API 访问**
   - OpenAI API 密钥或其他提供商（请参阅 [文档](https://docs.cognee.ai/setup-configuration/llm-providers)）

4. **数据库配置**
   - 默认情况下不需要配置。Cognee 使用基于文件的数据库（LanceDB 和 Kuzu）
   - 可选地，您可以设置 Azure AI Search 作为向量存储（请参阅 [文档](https://github.com/topoteretes/cognee-community/tree/main/packages/vector/azureaisearch)）

### 环境配置

在项目目录中创建一个 `.env` 文件，包含以下变量：

```ini
# LLM 配置（必需）
LLM_API_KEY=your-openai-api-key-here

# 缓存配置（会话所需）
CACHING=true  # 会话历史记录必须启用

```


## 🏛️ 理解 Cognee 的记忆架构

### Cognee 如何工作

Cognee 提供了一个复杂的记忆系统，超越了简单的键值存储：

```
┌──────────────────────────┐
│      30+ 数据源          │
└───────────┬──────────────┘
            │
            ▼
┌──────────────────────────────────────────┐
│  动态演变的记忆层                          │
│                                          │
│  ┌────────────────────────────────────┐  │
│  │ 图数据库中的知识图谱                    │  │
│  └────────────────────────────────────┘  │
│  ┌────────────────────────────────────┐  │
│  │ 向量存储中的嵌入                      │  │
│  │   (例如，Azure AI Search)          │  │
│  └────────────────────────────────────┘  │
└───────────┬──────────────────────────────┘
            │                      ▲   
            ▼                      │(可选)
┌────────────────┐           ┌────────────────┐
│     cognee     │(可选) │ Cognee 会话      │
│    检索器      │──────────▶│     缓存        │
│                │           │    (Redis)     │
└───────┬────────┘           └────────────────┘
        ▲
        │
┌──────────────────────────┐
│          智能体           │
└──────────────────────────┘

```

### 关键组件：

1. **知识图谱**：存储实体、关系和语义连接
2. **向量嵌入**：启用所有存储信息的语义搜索
3. **会话缓存**：维护会话内和跨会话的对话上下文
4. **NodeSets**：将数据组织成逻辑类别以进行有针对性的检索

### 本教程中的记忆类型：

- **持久记忆**：知识图谱中的长期存储
- **会话记忆**：Redis 缓存中的临时对话上下文
- **语义记忆**：基于向量的所有数据的相似性搜索

## 📦 安装所需包

安装支持会话管理的 Cognee：

In [ ]:
!pip install --quiet "cognee[redis]==0.4.0"

## 🔧 初始化环境并加载库

确保：
1. Redis 正在运行（例如，通过 Docker：`docker run -d -p 6379:6379 redis`）
2. 在导入缓存模块之前设置环境变量
3. 如有需要，重启内核并按顺序运行单元格

以下单元格将：
1. 从 `.env` 加载环境变量
2. 使用您的 LLM 设置配置 Cognee
3. 启用会话管理的缓存
4. 验证所有组件是否正确连接

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

# cognee 配置
os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", None)
os.environ["CACHING"] = os.getenv("CACHING", "true")


import cognee

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")
print(f"LLM_API_KEY: {os.environ.get('LLM_API_KEY')}")

## 📁 配置存储目录

Cognee 使用两个单独的目录进行操作：
- **数据根目录**：存储摄取的文档和处理的数据
- **系统根目录**：包含知识图谱数据库和系统元数据

我们将为本教程创建隔离的目录如下：

In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

print(f"Data root: {DATA_ROOT}")
print(f"System root: {SYSTEM_ROOT}")

## 🧹 重置记忆状态

在开始构建我们的记忆系统之前，让我们确保我们是从全新状态开始的。

> 💡 **提示**：如果您希望在以后使用此笔记本时保留之前运行的现有记忆，可以跳过此步骤。

In [ ]:
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print('Cleared previous Cognee state.')

## 📚 第 1 部分：构建知识库

### 我们开发者助手的数据源

我们将摄取三种类型的数据来创建全面的知识库：

1. **开发者资料**：个人专业知识和技术背景
2. **Python 最佳实践**：Python 之禅与实用指南
3. **历史对话**：开发者和 AI 助手之间的过去问答会话

这种多样化的数据使我们的智能体能够：
- 理解用户的技术背景
- 在推荐中应用最佳实践
- 从以前成功的交互中学习

In [ ]:
developer_intro = (
  "Hi, I'm an AI/Backend engineer. "
  "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
  "and production testing via pytest-asyncio. "
  "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = (
  """
    # The Zen of Python: Practical Guide

    ## Overview
    Use these principles as a checklist during design, coding, and reviews.

    ## Key Principles With Guidance

    ### 1. Beautiful is better than ugly
    Prefer descriptive names, clear structure, and consistent formatting.

    ### 2. Explicit is better than implicit
    Be clear about behavior, imports, and types.
    ```python
    from datetime import datetime, timedelta

    def get_future_date(days_ahead: int) -> datetime:
        return datetime.now() + timedelta(days=days_ahead)
    ```

    ### 3. Simple is better than complex
    Choose straightforward solutions first.

    ### 4. Complex is better than complicated
    When complexity is needed, organize it with clear abstractions.

    ### 5. Flat is better than nested
    Use early returns to reduce indentation.

    ## Modern Python Tie-ins
    - Type hints reinforce explicitness
    - Context managers enforce safe resource handling
    - Dataclasses improve readability for data containers

    ## Quick Review Checklist
    - Is it readable and explicit?
    - Is this the simplest working solution?
    - Are errors explicit and logged?
    - Are modules/namespaces used appropriately?
  """
)

human_agent_conversations = (
  """
  "conversations": [
      {
        "id": "conv_001",
        "timestamp": "2024-01-15T10:30:00Z",
        "topic": "async/await patterns",
        "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
        "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, context managers for session lifecycle, and robust exception handling for failed requests.",
        "code_context": {
          "file": "scraper.py",
          "language": "python",
          "patterns_discussed": ["async/await", "context_managers", "semaphores", "aiohttp", "error_handling"]
        },
        "follow_up_questions": [
          "How do I add retry logic for failed requests?",
          "What's the best way to parse the scraped HTML content?"
        ]
      },
      {
        "id": "conv_002",
        "timestamp": "2024-01-16T14:20:00Z",
        "topic": "dataclass vs pydantic",
        "user_query": "When should I use dataclasses vs Pydantic models? I'm building an API and need to handle user input validation.",
        "assistant_response": "For API input/output, prefer Pydantic: it provides runtime validation, type coercion, JSON serialization, enums for roles, field constraints, and custom validators; integrates cleanly with FastAPI for automatic request validation and error reporting.",
        "code_context": {
          "file": "models.py",
          "language": "python",
          "patterns_discussed": ["pydantic", "dataclasses", "validation", "fastapi", "type_hints", "enums"]
        },
        "follow_up_questions": [
          "How do I handle nested validation with Pydantic?",
          "Can I use Pydantic with SQLAlchemy models?"
        ]
      },
      {
        "id": "conv_003",
        "timestamp": "2024-01-17T09:15:00Z",
        "topic": "testing patterns",
        "user_query": "I'm struggling with testing async code and database interactions. What's the best approach for pytest with async functions?",
        "assistant_response": "Recommended using pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async functions and database interactions in FastAPI.",
        "code_context": {
          "file": "test_users.py",
          "language": "python",
          "patterns_discussed": ["pytest", "async_testing", "fixtures", "mocking", "database_testing", "fastapi_testing"]
        },
        "follow_up_questions": [
          "How do I test WebSocket connections?",
          "What's the best way to test database migrations?"
        ]
      },
      {
        "id": "conv_004",
        "timestamp": "2024-01-18T16:45:00Z",
        "topic": "performance optimization",
        "user_query": "My FastAPI app is getting slow with large datasets. How can I optimize database queries and response times?",
        "assistant_response": "Suggested optimizing database queries (indexes, pagination, selecting only needed columns), adding caching, streaming or chunked responses for large datasets, background tasks for heavy work, and monitoring to find bottlenecks.",
        "code_context": {
          "file": "optimizations.py",
          "language": "python",
          "patterns_discussed": ["performance_optimization", "caching", "database_optimization", "async_patterns", "monitoring"]
        },
        "follow_up_questions": [
          "How do I implement database connection pooling properly?",
          "What's the best way to handle memory usage with large datasets?"
        ]
      },
      {
        "id": "conv_005",
        "timestamp": "2024-01-19T11:30:00Z",
        "topic": "error handling and logging",
        "user_query": "I need to implement proper error handling and logging across my Python application. What's the best approach for production-ready error management?",
        "assistant_response": "Proposed centralized error handling with custom exceptions, structured logging, FastAPI middleware or decorators, and integration points for external monitoring/alerting tools.",
        "code_context": {
          "file": "error_handling.py",
          "language": "python",
          "patterns_discussed": ["error_handling", "logging", "exceptions", "middleware", "decorators", "fastapi"]
        },
        "follow_up_questions": [
          "How do I integrate this with external monitoring tools like Sentry?",
          "What's the best way to handle errors in background tasks?"
        ]
      }
    ],
    "metadata": {
      "total_conversations": 5,
      "date_range": "2024-01-15 to 2024-01-19",
      "topics_covered": [
        "async/await patterns",
        "dataclass vs pydantic",
        "testing patterns",
        "performance optimization",
        "error handling and logging"
      ],
      "code_patterns_discussed": [
        "asyncio", "aiohttp", "semaphores", "context_managers",
        "pydantic", "fastapi", "type_hints", "validation",
        "pytest", "async_testing", "fixtures", "mocking",
        "performance_optimization", "caching", "database_optimization",
        "error_handling", "logging", "exceptions", "middleware"
      ],
      "difficulty_levels": {
        "beginner": 1,
        "intermediate": 2,
        "advanced": 2
      }
    }
  """
)

## 🔄 将数据处理为知识图谱

现在我们将把原始文本转换为结构化记忆。此过程：

1. **将数据添加到 NodeSets**：将信息组织到逻辑类别中
   - `developer_data`：开发者资料和对话
   - `principles_data`：Python 最佳实践和指南

2. **运行 Cognify 管道**：提取实体、关系并创建嵌入
   - 识别关键概念
   - 在相关信息之间创建语义连接
   - 生成向量嵌入

这可能需要几分钟时间，因为 LLM 处理文本并构建图结构：

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])await cognee.add(human_agent_conversations, node_set=["developer_data"])await cognee.add(python_zen_principles, node_set=["principles_data"])
await cognee.cognify()

## 📊 可视化知识图谱

让我们探索知识图谱的结构。可视化显示：
- **节点**：从文本中提取的实体（概念、技术、人员）
- **边**：实体之间的关系和连接
- **集群**：按语义相似性分组的相关概念

在浏览器中打开生成的 HTML 文件以交互式探索图谱：

In [ ]:
from cognee import visualize_graph
await visualize_graph('./visualization_1.html')

## 🧠 使用 Memify 丰富记忆

`memify()` 函数分析知识图谱并生成有关数据的智能规则。此过程：
- 识别模式和最佳实践
- 基于内容创建可操作的指导方针
- 建立不同知识领域之间的关系

这些规则帮助智能体在回答问题时做出更明智的决策。捕获第二个可视化有助于您比较图谱在丰富后的密集程度。


In [ ]:
await cognee.memify()

await visualize_graph('./visualization_2.html')

## 🔍 第 2 部分：智能记忆检索

### 演示 1：跨文档知识集成

现在我们的知识图谱已经构建完成，让我们测试 Cognee 如何结合多个来源的信息来回答复杂问题。

第一个查询演示：
- **语义理解**：即使未明确提及，也能找到相关概念
- **交叉引用**：将开发者资料与 Python 原则相结合
- **上下文推理**：将最佳实践应用于特定实现

### 演示 2：使用 NodeSets 进行过滤搜索

第二个查询显示如何针对知识图谱的特定子集：
- 使用 `node_name` 参数仅在 `principles_data` 中搜索
- 提供来自特定知识域的聚焦答案
- 当您需要特定领域信息时非常有用

In [ ]:
# demonstrate cross-document knowledge retrieval from multiple data sources
from cognee.modules.search.types import SearchType

results = await cognee.search(
    query_text="How does my AsyncWebScraper implementation align with Python's design principles?",
    query_type=SearchType.GRAPH_COMPLETION,
)
print("Python Pattern Analysis:", results)

# demonstrate filtered search using NodeSet to query only specific subsets of memory
from cognee.modules.engine.models.node_set import NodeSet
results = await cognee.search(
    query_text="How should variables be named?",
    query_type=SearchType.GRAPH_COMPLETION,
    node_type=NodeSet,
    node_name=["principles_data"],
)
print("Filtered search result:", results)

## 🔐 第 3 部分：会话管理设置

### 启用对话记忆

会话管理对于在多次交互中维护上下文至关重要。在这里我们将：

1. **初始化用户上下文**：创建或检索用于会话跟踪的用户配置文件
2. **配置缓存引擎**：连接到 Redis 以存储对话历史
3. **启用会话变量**：设置在查询之间持久存在的上下文变量

> ⚠️ **重要**：这需要 Redis 运行且环境中设置了 `CACHING=true`

In [ ]:
from cognee.modules.users.methods import get_default_user
from cognee.context_global_variables import set_session_user_context_variable 
from cognee.infrastructure.databases.cache import get_cache_engine

user = await get_default_user()
await set_session_user_context_variable(user)
print(f"Using user id: {getattr(user, 'id', 'unknown')}")

cache_engine = get_cache_engine()
if cache_engine is None:
    raise RuntimeError('Cache engine is not available. Double-check your cache configuration.')
print('Session cache is ready.')


## 🛠️ 辅助函数：查看会话历史

此实用函数允许我们检查存储在 Redis 中的对话历史。它对于：
- 调试会话管理
- 验证对话是否被缓存
- 了解智能体可用的上下文

In [ ]:
async def show_history(session_id: str) -> None:
    # Let's check the cache directly
    cache_engine = get_cache_engine()
    if cache_engine:
        # Try to get history directly from cache
        user_id = str(user.id) if hasattr(user, 'id') else None
        if user_id:
            history_entries = await cache_engine.get_latest_qa(user_id, session_id, last_n=10)
            print(f"\nDirect cache query for user_id={user_id}, session_id={session_id}:")
            print(f"Found {len(history_entries)} entries")
            if history_entries:
                for i, entry in enumerate(history_entries, 1):
                    print(f"\nEntry {i}:")
                    print(f"  Question: {entry.get('question', 'N/A')[:100]}...")
                    print(f"  Answer: {entry.get('answer', 'N/A')[:100]}...")
        else:
            print("No user_id available")


## 会话 1：异步支持实验室 — 第一个问题

通过询问大规模网页抓取器的遥测友好 asyncio 模式来启动 `async-support-lab` 会话。图谱已经了解 asyncio、aiohttp 和监控实践，因此响应应该反映之前的对话，同时根据新查询定制答案。


In [ ]:
session_1 = "async-support-lab"

result = await cognee.search(
    query_type=SearchType.GRAPH_COMPLETION,
    query_text="I'm building a web scraper that hits thousands of URLs concurrently. What's a reliable asyncio pattern with telemetry?",
    session_id=session_1
)

## 首次交换后检查会话 1 记忆

在初始问题后立即运行 `show_history(session_1)` 确认 Cognee 将提示和完成内容都写入了 Redis。您应该看到一个包含并发指导的条目。


In [ ]:
await show_history(session_1)

## 会话 1：关于数据模型的后续问题

接下来我们使用相同的会话 ID 询问："When should I pick dataclasses versus Pydantic?"。Cognee 应该将 Python 原则与之前的 FastAPI 对话结合起来，提供细致的建议 - 展示上下文在命名会话中如何延续。


In [ ]:
result = await cognee.search(
    query_type=SearchType.GRAPH_COMPLETION,
    query_text="When should I pick dataclasses versus Pydantic for this work?",
    session_id=session_1
)

## 确认会话 1 历史包含两个回合

另一个 `show_history(session_1)` 调用应该列出两个问答条目。这与 Mem0 实验室的"记忆回放"步骤相匹配，证明额外的回合会扩展同一个 transcript。


In [ ]:
await show_history(session_1)

## 会话 2：设计审查线程 — 新会话

为了展示线程之间的隔离，我们启动 `design-review-session` 并请求事件审查的日志记录指南。尽管底层知识库相同，但新的会话 ID 使 transcript 保持分离。


In [ ]:
session_2 = "design-review-session"

result = await cognee.search(
    query_type=SearchType.GRAPH_COMPLETION,
    query_text="We're drafting logging guidance for incident reviews. Capture the key principles please.",
    session_id=session_2
)

## 审查会话 2 历史

`show_history(session_2)` 应该只列出设计审查提示/响应对。将其与会话 1 进行比较，以突出 Cognee 如何在重用共享知识图谱的同时保持独立 transcript。


In [ ]:
await show_history(session_2)

## 总结 

恭喜！您刚刚为编码助手提供了一个由 Cognee 提供支持的真正长期记忆层。

在本教程中，您将原始开发者内容（代码、文档、聊天）转换为图形 + 向量记忆，您的智能体可以搜索、推理并不断改进。

您学到了什么

1. **从原始文本到 AI 记忆**：Cognee 如何摄取非结构化数据并使用组合向量 + 知识图谱架构将其转变为智能、可搜索的记忆。

2. **使用 memify 丰富图谱**：如何超越基本图谱创建，并使用 memify 在现有图谱之上添加派生事实和更丰富的关系。 

3. **多种搜索策略**：如何根据智能体的需要使用不同的搜索类型（图谱感知问答、RAG 风格完成、见解、原始块、代码搜索等）查询记忆。 

4. **可视化探索**：如何使用图谱可视化和 Cognee UI 检查和调试 Cognee 构建的内容，以便您可以实际看到知识的结构。 

5. **会话感知记忆**：如何将会话特定上下文与持久语义记忆相结合，使智能体能够跨运行记住，同时不会在用户之间泄露信息。 

## 关键要点
1. 记忆作为由嵌入支持的知识图谱

    - **结构化理解**：Cognee 结合了向量存储和图形存储，使您的数据既能通过含义搜索，又能通过关系连接。Cognee 默认使用基于文件的数据库（LanceDB 用于向量，Kuzu 用于图形数据库）

    - **关系感知检索**：答案不仅可以基于"相似文本"，还可以基于实体如何相关。

    - **活记忆**：记忆层不断发展、增长，并作为一个连接的图形保持可查询性。 

2. 搜索和推理模式
    - **混合检索**：搜索融合了向量相似性、图形结构和 LLM 推理，从原始块查找��到图形感知问答。 

    - **根据任务选择模式**：当您需要自然语言答案时使用完成式模式，当您的智能体需要原始上下文或驱动自己的推理时使用块/摘要/图形模式。

3. 个性化、会话感知智能体
    - **会话上下文 + 长期记忆**：Cognee 将短期"线程"上下文与长期、用户或组织级别的记忆分开。 

## 实际应用

1. **垂直 AI 智能体**

    使用本笔记本中的模式为领域智能副驾驶提供动力，这些副驾驶位于 Cognee 之上作为其检索和推理核心：

- **开发者副驾驶**：代码审查、事件分析和架构助手，将代码、API、设计文档和工单作为单个记忆图谱遍历。

- **面向客户的副驾驶**：支持或成功智能体，通过图形感知检索和引用答案从产品文档、FAQ、CRM 笔记和过去的工单中提取信息。

- **内部专家副驾驶**：政策、法律或安全助手，对相互关联的规则、指南和历史决策进行推理，而不是孤立的 PDF。

    Cognee 明确定位为 AI 智能体的持久、准确记忆，提供一个活知识图谱，位于您的智能体后面，取代临时的向量存储和自定义图形代码组合。 

2. **将数据孤岛统一到一个记忆中**

    同样的方法也可以帮助您构建跨分散源的统一记忆层：

- **从孤岛到一个图形**：将结构化（如数据库）和非结构化数据（如文档、聊天）摄取到由嵌入支持的单个图形中，而不是每个系统的单独索引。 

- **带引用的跨源推理**：对所有内容运行多步推理 - 通过图形"连接"日志、指标和文档 - 并仍然返回带有来源的基础答案。 

- **知识中心**：对于银行或教育等领域，Cognee 已经被用于将 PDF、内部系统和应用数据统一到一个带有向量的知识图谱中，以便智能体可以使用精确、引用的上下文回答问题。 

## 后续步骤

您已经实现了核心记忆循环。以下是您可以自己尝试的自然扩展（有关详细信息，请参阅 [Cognee 文档](https://docs.cognee.ai/)）：

1. **实验时间感知**：打开时间 cognify 以从文本中提取事件和时间戳。

2. **引入本体驱动推理**：为您的域定义 OWL 本体。使用 Cognee 的本体支持，使提取的实体和关系基于该模式，提高图谱质量和域特定答案。 

3. **添加反馈循环**：让 Cognee 根据真实用户反馈调整图形边缘权重，以便检索随时间改进而不是保持静态。 

4. **为个性化和会话行为调整**：使用用户 ID、租户和数据集为每个人或团队提供对共享记忆引擎的自己的视图。 

5. **扩展到更复杂的智能体**：将 Cognee 插入到智能体框架中，构建多智能体系统，共享相同的记忆层。 *Microsoft Agent Framework x Cognee 插件即将推出。*